# 03 - Portfolio Targets

Use this notebook after generating momentum rankings. It walks down the ranked list and suggests buy quantities using the ATR-based formula:

`shares = AccountValue * 0.001 / ATR20`

The default behavior only buys a candidate when there is enough cash for the full suggested share count.

In [7]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from clenow.portfolio import build_buy_list

In [ ]:
# Portfolio inputs. Edit these before each weekly run.
ACCOUNT_VALUE = 100_000.00
AVAILABLE_CASH = 100_000.00
DAILY_MOVE_TARGET = 0.003  # 10 basis points of account value per ATR20 move
ALLOW_PARTIAL_FINAL_POSITION = False

# Optional: add current holdings if you want shares_to_buy to be a delta to target.
# Example: {"AAPL": 25, "MSFT": 10}
EXISTING_POSITIONS = {}

In [9]:
RANKINGS_PATH = PROJECT_ROOT / "data" / "processed" / "momentum_rankings.parquet"
BUY_LIST_PATH = PROJECT_ROOT / "data" / "processed" / "buy_list.csv"
DIAGNOSTICS_PATH = PROJECT_ROOT / "data" / "processed" / "buy_list_diagnostics.csv"

rankings = pd.read_parquet(RANKINGS_PATH)
print(f"Loaded {len(rankings)} ranked candidates")
rankings.head()

Loaded 204 ranked candidates


,rank,ticker,date,last_price,ATR20,MA100,avg_dollar_volume_20,annualized_slope,r_squared,momentum_score,above_trend_ma,history_days
0,1,SNDK,2026-05-22,1478.689941,116.352704,763.578999,2.034255e+10,24.187924,0.861570,20.839589,True,320
1,2,CIEN,2026-05-22,583.739990,35.836505,382.623100,1.083841e+09,14.889722,0.956468,14.241541,True,753
2,3,LITE,2026-05-22,946.900024,84.689005,674.612800,5.872106e+09,15.909667,0.835655,13.294998,True,753
3,4,DELL,2026-05-22,295.190002,15.020998,162.690373,1.523937e+09,11.864750,0.950771,11.280654,True,753
4,5,INTC,2026-05-22,119.839996,8.884001,61.140400,1.669334e+10,15.171299,0.656575,9.961096,True,753


In [10]:
buy_list, diagnostics = build_buy_list(
    rankings,
    account_value=ACCOUNT_VALUE,
    available_cash=AVAILABLE_CASH,
    daily_move_target=DAILY_MOVE_TARGET,
    atr_column="ATR20",
    existing_positions=EXISTING_POSITIONS,
    allow_partial_final_position=ALLOW_PARTIAL_FINAL_POSITION,
)

buy_list.to_csv(BUY_LIST_PATH, index=False)
diagnostics.to_csv(DIAGNOSTICS_PATH, index=False)

estimated_total = buy_list["estimated_cost"].sum() if not buy_list.empty else 0.0
remaining_cash = AVAILABLE_CASH - estimated_total
print(f"Suggested {len(buy_list)} buys")
print(f"Estimated spend: ${estimated_total:,.2f}")
print(f"Remaining cash: ${remaining_cash:,.2f}")
buy_list.head(50)

Suggested 12 buys
Estimated spend: $95,519.08
Remaining cash: $4,480.92


,rank,ticker,date,last_price,ATR20,MA100,avg_dollar_volume_20,annualized_slope,r_squared,momentum_score,above_trend_ma,history_days,target_shares,shares_to_buy,estimated_cost,remaining_cash
0,1,SNDK,2026-05-22,1478.689941,116.352704,763.578999,2.034255e+10,24.187924,0.861570,20.839589,True,320,4,4,5914.759766,94085.240234
1,2,CIEN,2026-05-22,583.739990,35.836505,382.623100,1.083841e+09,14.889722,0.956468,14.241541,True,753,13,13,7588.619873,86496.620361
2,3,LITE,2026-05-22,946.900024,84.689005,674.612800,5.872106e+09,15.909667,0.835655,13.294998,True,753,5,5,4734.500122,81762.120239
3,4,DELL,2026-05-22,295.190002,15.020998,162.690373,1.523937e+09,11.864750,0.950771,11.280654,True,753,33,33,9741.270081,72020.850159
4,5,INTC,2026-05-22,119.839996,8.884001,61.140400,1.669334e+10,15.171299,0.656575,9.961096,True,753,56,56,6711.039795,65309.810364
5,6,STX,2026-05-22,812.729980,49.380496,475.389870,3.529277e+09,9.336687,0.751424,7.015808,True,753,10,10,8127.299805,57182.510559
6,7,WDC,2026-05-22,484.279999,30.912003,314.856968,3.662958e+09,7.118080,0.831811,5.920896,True,753,16,16,7748.479980,49434.030579
7,8,VRT,2026-05-22,327.459991,18.900700,256.429684,1.930811e+09,6.095813,0.891507,5.434463,True,753,26,26,8513.959778,40920.070801
8,9,COHR,2026-05-22,377.570007,27.213503,265.652301,2.305883e+09,5.225678,0.863899,4.514457,True,753,18,18,6796.260132,34123.810669
9,10,GLW,2026-05-22,194.050003,13.788998,139.095191,2.875679e+09,5.252102,0.812214,4.265832,True,753,36,36,6985.800110,27138.010559


In [11]:
display_columns = [
    "rank",
    "ticker",
    "last_price",
    "ATR20",
    "momentum_score",
    "target_shares",
    "shares_to_buy",
    "estimated_cost",
    "remaining_cash",
]
buy_list[display_columns] if not buy_list.empty else buy_list

,rank,ticker,last_price,ATR20,momentum_score,target_shares,shares_to_buy,estimated_cost,remaining_cash
0,1,SNDK,1478.689941,116.352704,20.839589,4,4,5914.759766,94085.240234
1,2,CIEN,583.739990,35.836505,14.241541,13,13,7588.619873,86496.620361
2,3,LITE,946.900024,84.689005,13.294998,5,5,4734.500122,81762.120239
3,4,DELL,295.190002,15.020998,11.280654,33,33,9741.270081,72020.850159
4,5,INTC,119.839996,8.884001,9.961096,56,56,6711.039795,65309.810364
5,6,STX,812.729980,49.380496,7.015808,10,10,8127.299805,57182.510559
6,7,WDC,484.279999,30.912003,5.920896,16,16,7748.479980,49434.030579
7,8,VRT,327.459991,18.900700,5.434463,26,26,8513.959778,40920.070801
8,9,COHR,377.570007,27.213503,4.514457,18,18,6796.260132,34123.810669
9,10,GLW,194.050003,13.788998,4.265832,36,36,6985.800110,27138.010559


In [12]:
diagnostics.head(100)

,ticker,rank,target_shares,shares_to_buy,estimated_cost,remaining_cash,action
0,SNDK,1,4,4,5914.759766,94085.240234,buy
1,CIEN,2,13,13,7588.619873,86496.620361,buy
2,LITE,3,5,5,4734.500122,81762.120239,buy
3,DELL,4,33,33,9741.270081,72020.850159,buy
4,INTC,5,56,56,6711.039795,65309.810364,buy
...,...,...,...,...,...,...,...
95,LNT,96,397,397,29358.148788,4480.920654,insufficient_cash
96,HWM,97,57,57,14623.349304,4480.920654,insufficient_cash
97,NI,98,570,570,27274.499130,4480.920654,insufficient_cash
98,SBAC,99,91,91,18706.870667,4480.920654,insufficient_cash
